In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

C:\Users\Lee\AppData\Local\Temp\ipykernel_11824\199085109.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader


True

In [4]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna")

# 벡터 저장소 설정
# 임베딩 및 저장

# 경로
DB_PATH = "../data/chroma_3store"
csv_path = "../data/16-6_개인정보FAQ.csv"
pdf_paths = [
    "../data/16-1_K희망사다리2026_모두의정책.pdf",
    "../data/Samsung_Electronics_Sustainability_Report_2026_KOR.pdf",
]

# CSV 로드
csv_docs = CSVLoader( # csv는 dict와 다르게 encoding 인자를 받지 않음
    file_path=csv_path,
    encoding="utf-8-sig", #cp949는 csv_read이고 현재는 
).load()


# PDF 로드
pdf_docs = []

for pdf_path in pdf_paths:
    pdf_docs.extend(
        PyPDFLoader(pdf_path).load()
    )

# 전체 문서 합치기
documents = pdf_docs + csv_docs

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 질문과 가까운 부분만 찾을 가능성 높이기 위해 청크 사용 -  관련 내용 조각화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
split_documents = text_splitter.split_documents(documents)

# 벡터 검색은 문서전체 X, 질문과 가장 비슷하고 작은
# 검색결과 너무 많은 내용을 포함 -> 청크 작게
# 답변 시 필요 문맥이 끊긴다 -> 청크 크게

In [10]:
# PDF 1 
pdf1_docs = PyPDFLoader(
    str(pdf_paths[0])
).load()

for doc in pdf1_docs:
    doc.metadata.update({
        "source_type": "pdf",
        "source_file": Path(pdf_paths[0]).name, #Path 라이브러리 필요
        "document_id": "policy_hope_ladder",
    })



In [ ]:
#   print(pdf_paths[0]) # 해당 컬럼 존재하는 지 확인

../data/16-1_K희망사다리2026_모두의정책.pdf


In [11]:
# PDF 2
pdf2_docs = PyPDFLoader(
    str(pdf_paths[1])
).load()

for doc in pdf2_docs:
    doc.metadata.update({
        "source_type": "pdf",
        "source_file": Path(pdf_paths[1]).name,
        "document_id": "samsung_sustainability",
    })

In [12]:
# csv
for row_number, doc in enumerate(csv_docs): #
    doc.metadata.update({
        "source_type": "csv",
        "source_file": Path(csv_path).name,
        "document_id": "privacy_faq",
        "row": row_number,
    })

In [ ]:
from pypdf import PdfReader

# reader = PdfReader("")
total_pages = len(reader.pages)
total_pages

In [ ]:
# 페이지 확인
reader.pages[1]

text = reader.pages[3].extract_text()
print(text)

In [8]:
# Document 형태로 정리 ---> 3가지
pdf_docs = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(Document(page_content=text,
                             metadata={"source" : "Samsung_Electronics",
                                       "page" : i + 1}
                             ))
pdf_docs[:10]

NameError: name 'reader' is not defined

In [ ]:
문서 청킹

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400,
                                          chunk_overlap=80
                                          )
chunks = splitter.split_documents(pdf_docs)
len(chunks)

1216

In [ ]:
# DB_PATH = "해당경로"
# COLLECTION = "해당 파일"


# #벡터 DB 생성 및 저장  --> 돈이 나가서 1번만 실행
# vectorstore = Chroma.from_documents(
#     documents= split_documents,  #- split_documents: PDF와 CSV에서 만든 분할 문서
#     embedding=embeddings, #
#     collection_name="3store", #
#     persist_directory=str(DB_PATH), # 
# )

#저장된 벡터 DB 불러오기
load_vs = Chroma(
    collection_name="3store", #
    embedding_function=embeddings, #
    persist_directory=str(DB_PATH), #
)

# # 저장된 벡터 DB 가져와야
# load_vs = Chroma(
#     collection_name="3store",
#     embedding_function=embeddings,
#     persist_directory=DB_PATH
# )

In [13]:
documents = pdf1_docs + pdf2_docs + csv_docs
print("전체 문서 수:", len(documents))

전체 문서 수: 457


In [37]:
# print("PDF 문서 수:", len(pdf_docs))
# print("CSV 문서 수:", len(csv_docs))
print(pdf_docs[0].metadata)
print(csv_docs[0])

{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.4 (Macintosh)', 'creationdate': '2026-02-09T17:43:27+09:00', 'moddate': '2026-03-03T16:13:53+09:00', 'trapped': '/False', 'source': '../data/16-1_K희망사다리2026_모두의정책.pdf', 'total_pages': 268, 'page': 0, 'page_label': '1'}
page_content='처리상황단계내용: 제공
적용분야내용: 금융 분야
개인정보유형내용: 일반정보
코드제목: 채무보증인의 상속인에게 채무자의 개인정보 제공 가능?
주제내용: 채무보증인의 상속인에게 원 채무자의 개인정보 제공 정당성
문제상황내용: A는 직무수행 중 부상을 당해 전역하였으며, 전역 후 사업을 하기위해 국가유공자 신분으로 B를 보증인으로 하여 은행으로부터 유리한 조건으로 대출을 받았습니다. A는 어느 날부터 원리금을 미납하기 시작하여 은행은 전화 및 우편 등을 통해 A에게 연락을 취하였으나 전화번호는 변경되었고, 발송된 우편물은 수신자 미거주로 반송되었습니다. 이후 A의 보증인 B마저 사망하자 은행은 B의 상속인에게 A의 대부원리금 상환을 요구하며 부동산과 예금에 가압류 조치를 할 수 있음을 통보하였습니다. B의 상속인은 대부원리금 상환 요구에 대한 억울함을 호소하며 은행에게 채무자 A의 연락처를 알려달라고 합니다.
질문: 상속인의 요청에 따라 은행은 채무자 A의 연락처(변경 전 전화번호, 우편물 반송된 주소 등을 말함)를 상속인에게 제공할 수 있는지요?
해결방법내용: 개인정보보호법에 따라 개인정보처리자(은행)는 정보주체 또는 제3자의 이익을 부당하게 침해할 우려가 있을 때를 제외하고 정보주체 또는 그 법정대리인이 주소불명 등으로 사전 동의를 받을 수 없는 경우로, 명백히 정보주체 또는 제3자의 급박한 생명, 신체, 재산의 이익을 위하여 

In [ ]:
# 문서 청킹

In [ ]:
# 검색기
retreiver_mmr = load_vs.as_retriever(search_type="mmr",
                                     search_kwargs={"k": 5, 
                                                    "fetch_k": 50,
                                                    "lambda_mult" : 0.25})
retreiver_mmr.invoke("삼성 전자 주가가 올라갈 타이밍 확인")

In [ ]:
RAG 구현

In [25]:
# RAG 로 붙여보기 - 정답이 없다
# RAG 로 붙여보기
SYSTEM_PROMPT = """
너는 삼성전자 주주를 위한 지속가능경영 및 기업 정보 안내 도우
미다.

사용자는 삼성전자의 경영성과, 지속가능경영 활동, 환경·사회·지
배구조(ESG),
사업부문, 재무 관련 수치와 기업 리스크에 대해 질문할 수 있다.

다음 원칙을 반드시 지켜라.

1. 답변은 반드시 참고 자료에 근거하여 작성하라.
2. 참고 자료에 없는 내용은 추측하거나 만들어 내지 말고
    "제공된 자료에서는 확인할 수 없습니다."라고 답하라.
3. 주주가 이해하기 쉽도록 결론을 먼저 제시하고,
    필요한 경우 근거와 세부 내용을 설명하라.
4. 매출, 영업이익, 비율, 인원, 배출량, 목표 연도 등 수치는
    원문에 나온 값과 단위를 정확하게 유지하라.
5. 현재 성과와 미래 목표를 구분하여 설명하라.
    확정된 실적을 미래의 전망이나 약속처럼 표현하지 마라.
6. 회사의 성과뿐 아니라 관련된 한계, 위험, 전제조건이
    참고 자료에 제시되어 있다면 함께 설명하라.
7. 질문이 여러 사업부문이나 연도를 비교하는 내용이면
    기준을 명확히 밝히고 표 또는 항목별 형식으로 정리하라.
8. 자료에 서로 다른 연도나 기준의 수치가 함께 있으면
    각각의 기준 연도와 측정 기준을 반드시 표시하라.
9. 주가의 상승·하락, 매수·매도, 투자 적합성에 대한 판단이나
    개인적인 투자 조언은 하지 마라.
10. 질문이 투자 판단과 관련되어 있으면 객관적인 자료와 사실만
설명하고,
    투자 결정은 사용자의 판단이 필요하다고 안내하라.
11. 참고 자료의 metadata에 page 정보가 있으면 답변 마지막에
    "[출처: p.페이지번호]" 형식으로 표시하라.
12. 질문의 범위가 불명확하거나 비교 기준이 부족하면
    임의로 해석하지 말고 필요한 내용을 짧게 되물어라.

답변 형식:

- 핵심 답변
- 근거가 되는 주요 수치 또는 사실
- 주주가 확인해야 할 조건이나 한계
- 출처 페이지

참고 자료:
{context}
"""

In [26]:
# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 text 로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata['page']}] \n {doc.page_content} \n\n"

    return context

In [18]:
chain = retreiver_mmr | format_docs
chain

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000021EC59853A0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50, 'lambda_mult': 0.25})
| RunnableLambda(format_docs)

In [19]:
# Chain 확인용
chain.invoke("청년 월세 지원 정책 찾아줘")

'[p.96] \n 095모두의\t정책\tK-희망사다리\t2026\n사업명 지원대상 및 핵심내용 시기 이용방법(문의처)\n청년일자리 \n도약장려금\n•\t(사업내용)\t사업주\t및\t근로자를\t지원하여\t청년신규\t\n일자리\t창출을\t통한\t청년고용\t활성화\t도모\t\n\t\t\t\t-\t\t수도권:\t5인\t이상\t우선지원대상기업에서\t취업애로\t\n청년*을\t정규직으로\t채용하고\t6개월\t이상\t고용\n유지\t시\t최장\t1년간\t최대\t720만\t원\t지원\n\t\t\t\t\t\t\t*\t취업애로청년:\t만\t15~34세의\t▲4개월\t이상\t실업,\t▲\n고졸\t이하\t청년\t등\t\n\t\t\t\t-\t\t비수도권:\t5인\t이상\t우선지원대상기업·산업단지\t\n입주\t중견기업에서\t청년을\t정규직으로\t채용\t후\t6개월\t\n이상\t고용유지\t시\t최장\t1년간\t최대\t720만\t원을\t지원\n하고,\t해당\t기업에서\t6개월\t이상\t재직한\t청년에게\t\n2년간\t최대\t720만\t원*을\t지원\n\t\t\t\t\t\t\t*\t일반\t비수도권:\t480만\t원,\t우대지원지역:\t600만\t원,\t\n특별지원지역:\t720만\t원\n•(지원대상)\t\n\t\t\t\t-\t\t5인\t이상*\t우선지원대상기업·비수도권\t산업단지\t입주\t\n중견기업,\t해당\t기업\t취업\t청년\t등\n\t\t\t\t\t\t\t*\t지식서비스·문화콘텐츠·신재생에너지\t산업,\t청년\t창업\n기업,\t지역주력산업\t등은\t1인\t이상도\t가능\n\t\t\t\t-\t\t수도권:\t취업애로청년을\t채용한\t5인\t이상\t우선지원\n대상기업\n\t\t\t\t-\t\t비수도권:\t청년을\t채용한\t5인\t이상\t우선지원대상\n기업·산업단지\t입주\t중견기업\t및\t해당\t기업에\t취업\t\n후\t6개월\t이상\t재직한\t청년\n•(핵심내용)\t\n\t\t\t\t-\t\t수도권:\t기업에\t신규채용\t청년\t1인당\t월\t최대\t60만\t\n원씩\t최대\t1년간\t지

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# rag_prompt 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n질문{question}")
])

                            


'- **핵심 답변**  \n  삼성전자의 핵심 사업은 크게 **DX(Device eXperience) 부문**과 **DS(Device Solutions) 부문**으로 나뉩니다.\n\n- **근거가 되는 주요 수치 또는 사실**\n\n| 부문 | 주요 사업 | 매출 | 영업이익 |\n|---|---|---:|---:|\n| **DX** | 스마트폰, 네트워크 시스템, 컴퓨터, TV, 냉장고, 세탁기, 에어컨, 의료기기 등 완제품 | **187조 9,673억 원** | **12조 8,526억 원** |\n| **DS** | 메모리, Foundry, System LSI 등 반도체 부품 | **130조 1,281억 원** | **24조 8,580억 원** |\n\n  - **DX 부문**: 소비자가 사용하는 완제품을 생산·판매합니다.  \n  - **DS 부문**: 반도체 사업을 담당하며, **DRAM, NAND Flash, 모바일AP** 등을 생산·판매합니다. DS는 **메모리 사업, Foundry 사업, System LSI 사업**으로 구성됩니다.\n  - 삼성전자는 **2025년 말 기준 전 세계 221개의 제조사업장, 판매사업장, R&D 센터, 디자인 센터 등**을 보유하고 있습니다.\n\n- **주주가 확인해야 할 조건이나 한계**  \n  위 매출과 영업이익은 참고자료에 제시된 DX·DS 부문 수치이며, 자료의 해당 부분에는 별도의 기준 연도가 명시되어 있지 않습니다. 따라서 이를 특정 연도의 실적으로 단정하려면 추가적인 재무 기준 확인이 필요합니다.\n\n- **출처 페이지**  \n  [출처: p.4]'

In [32]:
# rangchain
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt  # question 은 그대로 prompt 에 들어가야
    | model
    | StrOutputParser()
    )

print(rag_chain.invoke("삼성 핵심 사업이 뭐지?"))

- **핵심 답변**  
  참고자료 기준으로 삼성전자 핵심 사업은 크게 **DX(Device eXperience) 부문**과 **DS(Device Solutions) 부문**입니다.  
  - **DX**: 스마트폰·TV·생활가전 등 완제품 사업  
  - **DS**: 메모리·파운드리·System LSI 등 반도체 사업

- **근거가 되는 주요 수치 또는 사실**

| 사업부문 | 주요 사업 | 매출 | 영업이익 |
|---|---|---:|---:|
| **DX** | 스마트폰, 네트워크 시스템, 컴퓨터, TV, 냉장고, 세탁기, 에어컨, 의료기기 등 | **187조 9,673억 원** | **12조 8,526억 원** |
| **DS** | 메모리, Foundry, System LSI 사업. DRAM, NAND Flash, 모바일AP 등 | **130조 1,281억 원** | **24조 8,580억 원** |

  삼성전자는 이 두 부문을 독립적으로 운영하며, **2025년 말 기준 전 세계 221개의 제조사업장·판매사업장·R&D 센터·디자인 센터 등**을 보유하고 있습니다.

- **주주가 확인해야 할 조건이나 한계**  
  제공된 자료의 사업부문별 매출·영업이익 수치는 원문에 제시된 값을 옮긴 것이며, 질문의 비교 기준이 되는 **구체적인 실적 기간은 해당 발췌문에서 확인할 수 없습니다.**  
  따라서 위 수치를 특정 회계연도의 실적이라고 단정하기보다는, 보고서에 제시된 DX·DS 부문별 수치로 확인하는 것이 적절합니다.

- **출처 페이지**  
  [출처: p.4]


In [ ]:
#구조화된 출력

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

# 구조화된 출력 받기
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    expected : str = Field(description="AI 예상 답변")

structured_model = model.with_structured_output(AnswerStyle, method="json_mode")
                            
rag_chain = (
    {"context" : ( retreiver_mmr | format_docs ), 
     "question" :  RunnablePassthrough()} # question 은 검색기를 거쳐서 문서찾아서 context 키값의 벨류 
    | rag_prompt
    | model   # question 은 그대로 prompt 에 들어가야
    | StrOutputParser() #structured_model
    )

result = rag_chain.invoke("삼성 핵심 사업이 뭐지?")
print(result)

- **핵심 답변**  
  삼성전자의 핵심 사업은 크게 **DX(Device eXperience) 부문**과 **DS(Device Solutions) 부문**으로 나뉩니다.

- **근거가 되는 주요 수치 또는 사실**

| 부문 | 주요 사업 | 매출 | 영업이익 |
|---|---|---:|---:|
| **DX** | 스마트폰, 네트워크 시스템, 컴퓨터, TV, 냉장고, 세탁기, 에어컨, 의료기기 등 완제품 | **187조 9,673억 원** | **12조 8,526억 원** |
| **DS** | 메모리, Foundry, System LSI 등 반도체 사업. DRAM, NAND Flash, 모바일AP 등 생산·판매 | **130조 1,281억 원** | **24조 8,580억 원** |

  - **DX 부문**은 소비자와 기업을 대상으로 스마트폰·TV·생활가전·의료기기 등 완제품을 생산·판매합니다.  
  - **DS 부문**은 메모리 반도체, 파운드리, 시스템 LSI 등 반도체 부품 사업을 담당합니다.  
  - 삼성전자는 **2025년 말 기준 전 세계 221개의 제조사업장, 판매사업장, R&D 센터, 디자인 센터 등**을 보유하고 있습니다.

- **주주가 확인해야 할 조건이나 한계**  
  - 위 매출과 영업이익의 **기준 연도와 세부 산출 기준은 제공된 자료에서 명확히 확인할 수 없습니다.** 다만 글로벌 네트워크 수치는 **2025년 말 기준**으로 제시되어 있습니다.  
  - 자료에서는 DX와 DS를 삼성전자의 2개 독립 운영 부문으로 설명하지만, 각 부문 내부 사업별 매출·영업이익은 제공되지 않았습니다.

- **출처 페이지**  
  [출처: p.4]
